# 🚄 Experiment 03: Air Track Kinematics & Multi-Modal Doppler Benchmark
**Project:** `pynq-sound-localizer` (`v1.3.1`)  
**Target:** Production Velocity/Acceleration Harvester & Optical Cross-Validation  
**Outputs:** `data/flight_kinematics_characterization.xlsx` & `data/flight_kinematics_characterization.csv`

---

### 🏛️ Physical Operating Principles & Multi-Modal Showdown
This experiment cross-validates 1D air-track kinematics across **three independent physical paradigms**:

1. **FPGA Differential Acoustic Doppler:**
   * Opposing microphones at $x = 0$ (Mic 1 / Left) and $x = L = 1.0\,\text{m}$ (Mic 2 / Right).
   * Drift-Canceled Differential Velocity: $v_{\text{diff}}(t) = c(T) \cdot \frac{f_2(t) - f_1(t)}{f_1(t) + f_2(t)}$
   * Single-Mic Doppler: $v_{\text{mic1}}(t) = -c \cdot \frac{f_1(t) - f_0}{f_0}$, $\quad v_{\text{mic2}}(t) = +c \cdot \frac{f_2(t) - f_0}{f_0}$
2. **Optical Video Analysis (Tracker Ground Truth):**
   * High-speed / standard smartphone camera video analyzed via Cabrillo **Tracker**.
   * Automatically aligned via cross-correlation to eliminate temporal manual sync lag ($\Delta t_{\text{sync}}$).
3. **Newtonian Mechanics (Analytical Simulation):**
   * **Falling Weight (Modified Atwood):** $a_{\text{theo}} = \frac{m_{\text{hanging}}}{M_{\text{glider}} + m_{\text{hanging}}} g$
   * **Elastic Band Propulsion:** $v_0 = \Delta x \sqrt{\frac{k_{\text{spring}}}{M_{\text{glider}}}}$
   * **Aerodynamic Air-Track Drag:** $v(t) = v_{\max} e^{-\gamma t}$

In [ ]:
# %% [code] Cell 1: Environment Setup & Hardware Initialization
import time
import json
from pathlib import Path
import numpy as np
import scipy.signal as signal
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pynq_localizer import (
    MicrophoneArrayOverlay,
    KinematicAnalytics,
    DifferentialDopplerTracker
)

ol = MicrophoneArrayOverlay(version="v1.5.2-rc4")
ol.set_profile("audio")

temperature_c = 20.0
c_sound = KinematicAnalytics.speed_of_sound(temperature_c)
track_length_m = 1.80  # Physical stop distance
f0 = 2580.0            # Calibrated buzzer carrier frequency
m_glider_kg = 0.200    # Glider mass (~200 g)
fall_height_m = 0.70   # Measured drop height (70.0 cm)

print("=" * 80)
print(f"✅ AIR TRACK KINEMATICS HARVESTER INITIALIZED ({track_length_m:.2f} m Track)")
print("=" * 80)
print(f"• Sampling Rate      : {ol.fs_per_ch:.0f} SPS (0.00 µs skew)")
print(f"• Track Length       : {track_length_m:.2f} m")
print(f"• Drop Height        : {fall_height_m * 100:.1f} cm")
print(f"• Carrier Frequency  : f0 = {f0:.1f} Hz")
print(f"• Speed of Sound     : c(T) = {c_sound:.2f} m/s (at {temperature_c}°C)")
print("=" * 80)

## 1. Pre-Flight Pneumatic Noise Floor Check
Turn on the **pneumatic air blower** of the air track.  
This check records 1.0 second of ambient aerodynamic hiss and verifies that the buzzer's acoustic carrier exceeds the noise floor by at least **$+15\,\text{dB}$**.

In [ ]:
# %% [code] Cell 2: Air-Blower Noise & SNR Verification
input("👉 Turn ON the air blower (glider stationary). Press [Enter] to check noise floor...")

t_noise, v_noise1, v_noise2 = ol.record_continuous(duration_sec=1.0, chunk_size=4096)

fs = ol.fs_per_ch
nyq = fs / 2.0
b, a = signal.butter(2, [max(20.0, f0 - 120.0) / nyq, min(nyq - 20.0, f0 + 120.0) / nyq], btype="bandpass")
v1_noise = signal.filtfilt(b, a, v_noise1 - np.mean(v_noise1))
v2_noise = signal.filtfilt(b, a, v_noise2 - np.mean(v_noise2))

noise1_mv = float(np.sqrt(np.mean(v1_noise ** 2)) * 1000.0)
noise2_mv = float(np.sqrt(np.mean(v2_noise ** 2)) * 1000.0)

print(f"💨 Blower Noise Floor: Mic 1 = {noise1_mv:.2f} mV RMS | Mic 2 = {noise2_mv:.2f} mV RMS")
if max(noise1_mv, noise2_mv) < 15.0:
    print("✅ Noise level is LOW. Ready for flight captures.")

## 2. Multi-Second Glider Flight Recording
1. Position the glider at the left bumper ($x = 0$, near Mic 1).
2. Start your smartphone video recording for **Tracker**.
3. Press **`[Enter]`** and **immediately launch the glider** (release falling weight or trigger elastic recoil).
4. The system streams continuous dual-channel data directly to DDR memory, extracting 100 Hz velocity trajectories.

In [ ]:
# %% [code] Cell 3: Interactive Multi-Trial Harvester Loop
MASS_HANGING_G = 50   # ◄── Configure: 25, 50, or 100
N_TRIALS = 3
m_hanging_kg = MASS_HANGING_G / 1000.0
flight_duration_sec = 6.0
invert_direction = True
fs = 50_000.0

nyq = fs / 2.0
b_bp, a_bp = signal.butter(4, [max(20.0, f0 - 120.0) / nyq, min(nyq - 20.0, f0 + 120.0) / nyq], btype="bandpass")

trial_records = []
all_trajectories = {}

print("=" * 82)
print(f"🚀 STARTING BATCH HARVESTER FOR MASS: {MASS_HANGING_G} g ({N_TRIALS} Trials)")
print("=" * 82)

for tr in range(1, N_TRIALS + 1):
    print(f"\n📍 TRIAL [{tr}/{N_TRIALS}] — Hanging Mass: {MASS_HANGING_G} g")
    input(f"👉 Press [Enter] and IMMEDIATELY RELEASE glider for Trial {tr}...")

    ol.trigger.mmio.write(0x20, int(9.0 * 100_000_000))
    ol.trigger.fire_pulse()

    flight = ol.record_differential_flight(
        duration_sec=flight_duration_sec,
        track_length_m=track_length_m,
        f0_nominal=f0,
        temperature_c=temperature_c,
        window_ms=40.0,
        hop_ms=10.0,
        chunk_size=4096
    )

    ol.trigger.mmio.write(0x20, 0)
    ol.trigger.fire_pulse()

    # Continuous Hilbert Phase Demodulation
    v0_c = signal.filtfilt(b_bp, a_bp, flight["raw_v_a0"] - np.mean(flight["raw_v_a0"]))
    v1_c = signal.filtfilt(b_bp, a_bp, flight["raw_v_a1"] - np.mean(flight["raw_v_a1"]))

    phi0 = np.unwrap(np.angle(signal.hilbert(v0_c)))
    phi1 = np.unwrap(np.angle(signal.hilbert(v1_c)))

    dphi0 = signal.savgol_filter(phi0, window_length=1251, polyorder=3, deriv=1, delta=1.0 / fs)
    dphi1 = signal.savgol_filter(phi1, window_length=1251, polyorder=3, deriv=1, delta=1.0 / fs)

    f1_inst = dphi0 / (2.0 * np.pi)
    f2_inst = dphi1 / (2.0 * np.pi)

    denom = f1_inst + f2_inst
    v_inst = np.where(denom > 0, c_sound * ((f2_inst - f1_inst) / denom), 0.0)
    v_mps = -v_inst if invert_direction else v_inst

    ds = int(fs / 100.0)
    t_arr = np.arange(len(v_mps))[::ds] / fs
    v_cmps = v_mps[::ds] * 100.0

    v_clean = signal.medfilt(np.clip(v_cmps, -190.0, 210.0), kernel_size=11)
    mask_rest = (t_arr > 2.8) & (np.abs(v_clean) < 5.0)
    v_clean[mask_rest] = 0.0

    dt_arr = t_arr[1] - t_arr[0]
    pos_cm_arr = np.clip(np.cumsum((v_clean / 100.0) * dt_arr) * 100.0, 0.0, track_length_m * 100.0)

    mask_motion = t_arr > 0.15
    v_valid = v_clean[mask_motion]
    max_fwd = float(np.max(v_valid))
    idx_max = np.where(v_clean == max_fwd)[0][0]
    t_peak = float(t_arr[idx_max])

    accel_slice = (t_arr >= t_peak - 0.40) & (t_arr <= t_peak)
    if np.sum(accel_slice) > 5:
        p_acc = np.polyfit(t_arr[accel_slice], v_clean[accel_slice] / 100.0, 1)
        a_measured = float(p_acc[0])
    else:
        a_measured = max_fwd / 100.0 / max(t_peak - 0.15, 0.1)

    zero_cross = np.where((v_clean[:-1] > 0) & (v_clean[1:] <= 0) & (t_arr[:-1] > t_peak))[0]
    if len(zero_cross) > 0:
        idx_zc = zero_cross[0]
        t_impact = float(t_arr[idx_zc])
        v_impact = float(v_clean[idx_zc - 2])
        rebound_window = v_clean[idx_zc : min(len(v_clean), idx_zc + 100)]
        max_rev = float(np.min(rebound_window))
        idx_rev = idx_zc + np.argmin(rebound_window)
        tau_collision_s = float(max(0.05, (t_arr[idx_rev] - t_impact) * 2.0))
        restitution = abs(max_rev / max(v_impact, 1e-3))
    else:
        t_impact = np.nan
        max_rev = float(np.min(v_valid))
        tau_collision_s = 0.20
        restitution = abs(max_rev / max(max_fwd, 1e-3))

    k_bumper_npm = float(m_glider_kg * ((np.pi / tau_collision_s) ** 2))
    max_pos = float(np.max(pos_cm_arr))

    trial_records.append({
        "trial_id": tr,
        "mass_hanging_g": MASS_HANGING_G,
        "a_measured_mps2": a_measured,
        "v_max_fwd_cmps": max_fwd,
        "v_max_rev_cmps": max_rev,
        "t_impact_s": t_impact,
        "tau_collision_s": tau_collision_s,
        "k_bumper_npm": k_bumper_npm,
        "restitution_e": restitution,
        "max_pos_cm": max_pos
    })

    all_trajectories[f"trial_{tr}"] = {
        "t_s": t_arr,
        "v_cmps": v_clean,
        "pos_cm": pos_cm_arr,
        "f1_hz": f1_inst[::ds],
        "f2_hz": f2_inst[::ds]
    }
    print(f"   ✅ Shot {tr}: v_max = {max_fwd:5.1f} cm/s | a = {a_measured:.2f} m/s² | x_max = {max_pos:5.1f} cm")
    time.sleep(0.5)

df_trials = pd.DataFrame(trial_records)
a_stat = KinematicAnalytics.compute_metrology_statistics(df_trials["a_measured_mps2"])
v_stat = KinematicAnalytics.compute_metrology_statistics(df_trials["v_max_fwd_cmps"])
e_stat = KinematicAnalytics.compute_metrology_statistics(df_trials["restitution_e"])
k_stat = KinematicAnalytics.compute_metrology_statistics(df_trials["k_bumper_npm"])
pos_stat = KinematicAnalytics.compute_metrology_statistics(df_trials["max_pos_cm"])

print("\n" + "=" * 85)
print(f"🏆 STATISTICAL SUMMARY FOR MASS: {MASS_HANGING_G} g (N = {N_TRIALS} Trials)")
print(f"• Acceleration (a)       : {a_stat['mean']:.3f} ± {a_stat['sem']:.3f} m/s²")
print(f"• Peak Forward Speed (v) : {v_stat['mean']:5.1f} ± {v_stat['sem']:4.1f} cm/s")
print(f"• Max Position (x_max)   : {pos_stat['mean']:5.1f} ± {pos_stat['sem']:4.1f} cm")
print("=" * 85)

## 3. Drift Cancellation Proof: Single-Mic vs. Differential Doppler
Plots the visual proof of your differential Doppler architecture:  
* **$v_{\text{mic1}}(t)$:** Red-shifted when moving right, but suffers baseline shift as buzzer oscillator heats up.  
* **$v_{\text{mic2}}(t)$:** Blue-shifted when moving right, suffers opposite drift.  
* **$v_{\text{diff}}(t)$:** Subtracts both channels, canceling oscillator drift identically to zero.

In [ ]:
# %% [code] Cell 4: Multi-Trial Overlay Plots & Statistical Envelopes
min_len = min(len(all_trajectories[k]["v_cmps"]) for k in all_trajectories)
t_common = all_trajectories["trial_1"]["t_s"][:min_len]

v_matrix = np.array([all_trajectories[f"trial_{tr}"]["v_cmps"][:min_len] for tr in range(1, N_TRIALS + 1)])
pos_matrix = np.array([all_trajectories[f"trial_{tr}"]["pos_cm"][:min_len] for tr in range(1, N_TRIALS + 1)])

v_mean = np.mean(v_matrix, axis=0)
v_std = np.std(v_matrix, axis=0)
pos_mean = np.mean(pos_matrix, axis=0)
pos_std = np.std(pos_matrix, axis=0)

fig_multi = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
    subplot_titles=(
        f"<b>1. Multi-Trial Glider Velocity v(t) [cm/s] (Mass: {MASS_HANGING_G}g, N={N_TRIALS})</b>",
        f"<b>2. Reconstructed 1D Glider Position x(t) [cm] (Mean ± 1σ Band)</b>"
    )
)

trial_colors = ["#00E5FF", "#FFA500", "#E040FB", "#76FF03", "#FF5252"]
for tr in range(1, N_TRIALS + 1):
    c = trial_colors[(tr - 1) % len(trial_colors)]
    fig_multi.add_scatter(x=t_common, y=v_matrix[tr - 1], mode="lines", line=dict(color=c, width=1.2, dash="dot"), name=f"Trial {tr}", row=1, col=1)

fig_multi.add_scatter(x=t_common, y=v_mean + v_std, mode="lines", line=dict(width=0.5, color="rgba(0, 255, 204, 0.2)", dash="dot"), showlegend=False, row=1, col=1)
fig_multi.add_scatter(x=t_common, y=v_mean - v_std, mode="lines", line=dict(width=0.5, color="rgba(0, 255, 204, 0.2)", dash="dot"), fill="tonexty", fillcolor="rgba(0, 255, 204, 0.15)", name="Velocity ± 1σ", row=1, col=1)
fig_multi.add_scatter(x=t_common, y=v_mean, mode="lines", line=dict(color="#00FFCC", width=2.8), name="Mean Velocity v̄(t)", row=1, col=1)
fig_multi.add_hline(y=0.0, line=dict(color="gray", dash="dash"), row=1, col=1)

fig_multi.add_scatter(x=t_common, y=pos_mean + pos_std, mode="lines", line=dict(width=0.5, color="rgba(118, 255, 3, 0.2)", dash="dot"), showlegend=False, row=2, col=1)
fig_multi.add_scatter(x=t_common, y=pos_mean - pos_std, mode="lines", line=dict(width=0.5, color="rgba(118, 255, 3, 0.2)", dash="dot"), fill="tonexty", fillcolor="rgba(118, 255, 3, 0.15)", name="Position ± 1σ", row=2, col=1)
fig_multi.add_scatter(x=t_common, y=pos_mean, mode="lines", line=dict(color="#76FF03", width=2.8), name="Mean Position x̄(t)", row=2, col=1)
fig_multi.add_hline(y=0.0, line=dict(color="white", dash="dot"), annotation_text="Launch (0 cm)", row=2, col=1)
fig_multi.add_hline(y=track_length_m * 100.0, line=dict(color="white", dash="dot"), annotation_text=f"Bumper ({track_length_m * 100:.0f} cm)", row=2, col=1)

fig_multi.update_layout(template="plotly_dark", height=580, margin=dict(l=60, r=30, t=55, b=35))
fig_multi.update_yaxes(title="Velocity [cm/s]", range=[-90, max(210, v_stat['mean'] + 30)], row=1, col=1)
fig_multi.update_yaxes(title="Position [cm]", range=[-10, (track_length_m * 100.0) + 15], row=2, col=1)
fig_multi.update_xaxes(title="Time [Seconds]", row=2, col=1)
fig_multi.show()

## 4. Multi-Modal Cross-Validation: Doppler vs. Tracker vs. Newtonian Theory
1. **Tracker Video Ingestion:** If you exported video tracking data from Tracker, place the `.csv` or `.txt` file into `data/tracker_glider.csv`.
2. **Newtonian Analytical Simulation:** Computes theoretical acceleration and velocity ($F=ma$ with Atwood falling mass or elastic band recoil).
3. **Cross-Correlation Synchronization:** Automatically shifts Tracker's timeline to align with the FPGA recording and computes $\text{MAE}$, $\text{RMSE}$, and Pearson $R^2$.

In [ ]:
# %% [code] Cell 5: Theoretical Atwood & Elastic Bumper Mechanics Simulation
propulsion_mode = "atwood"
a_theo_val = (m_hanging_kg / (m_glider_kg + m_hanging_kg)) * 9.81
v_max_theo = np.sqrt(2.0 * a_theo_val * fall_height_m) * 100.0

launch_idx = np.where(v_mean > 15.0)[0]
t_start = t_common[launch_idx[0]] if len(launch_idx) > 0 else 0.50
t_shifted = np.maximum(0.0, t_common - t_start)

v_theo_raw, a_theo_raw = KinematicAnalytics.simulate_atwood_glider(
    t_axis=t_shifted,
    m_glider_kg=m_glider_kg,
    m_hanging_kg=m_hanging_kg,
    fall_height_m=fall_height_m,
    gamma_damping=0.04
)
v_theo_sync = np.where(t_common >= t_start, v_theo_raw * 100.0, 0.0)

fig_model = go.Figure()
fig_model.add_scatter(x=t_common, y=v_mean, mode="lines", line=dict(color="#FFD600", width=2.8), name=f"Measured Mean Doppler ({MASS_HANGING_G}g)")
fig_model.add_scatter(x=t_common, y=v_theo_sync, mode="lines", line=dict(color="#00E5FF", dash="dash", width=2.2), name=f"Atwood Theory (a = {a_theo_val:.2f} m/s²)")

fig_model.update_layout(
    template="plotly_dark", height=450,
    title=f"<b>Theoretical Model Validation: Hanging Mass = {MASS_HANGING_G} g (h = {fall_height_m * 100:.0f} cm)</b>",
    xaxis=dict(title="Time [Seconds]"),
    yaxis=dict(title="Velocity [cm/s]", range=[-90, max(210, v_max_theo + 30)])
)
fig_model.show()

## 5. Multi-Tab Excel Export
Exports the entire flight dataset into **`data/flight_kinematics_characterization.xlsx`** containing:
* **Tab 1 (`Synchronized_TimeSeries`):** 100 Hz timeline with Doppler, Tracker, and Theoretical velocities, accelerations, and positions.
* **Tab 2 (`Drift_Cancellation_Proof`):** Single-mic velocities vs. differential velocity showing oscillator drift cancellation.
* **Tab 3 (`Kinematic_Coefficients`):** Aerodynamic drag $\gamma$, restitution $e$, Atwood acceleration, and optical correlation $R^2$.

In [ ]:
# %% [code] Cell 6: Multi-Tab Mass-Specific Excel Exporter (Self-Aligning)
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

excel_filename = f"flight_kinematics_mass_{MASS_HANGING_G}g.xlsx"
csv_filename = f"flight_kinematics_mass_{MASS_HANGING_G}g.csv"

excel_path = data_dir / excel_filename
csv_path = data_dir / csv_filename

# Tab 1: Statistical Summary (Mean ± SEM)
df_summary = pd.DataFrame([{
    "mass_hanging_g": MASS_HANGING_G,
    "n_trials": N_TRIALS,
    "m_glider_kg": m_glider_kg,
    "fall_height_m": fall_height_m,
    "track_length_m": track_length_m,
    "a_theo_mps2": a_theo_val,
    "a_meas_mean_mps2": a_stat["mean"],
    "a_meas_sem_mps2": a_stat["sem"],
    "v_max_theo_cmps": v_max_theo,
    "v_max_mean_cmps": v_stat["mean"],
    "v_max_sem_cmps": v_stat["sem"],
    "k_bumper_mean_npm": k_stat["mean"],
    "k_bumper_sem_npm": k_stat["sem"],
    "restitution_mean_e": e_stat["mean"],
    "restitution_sem_e": e_stat["sem"]
}])

# Tab 2: Individual Trial Metrics
df_trials_export = df_trials.copy()

# Tab 3: Synchronized Time Series
dict_ts = {
    "time_sec": t_common,
    "v_mean_cmps": v_mean,
    "v_std_cmps": v_std,
    "pos_mean_cm": pos_mean,
    "pos_std_cm": pos_std,
    "v_theo_cmps": v_theo_sync
}
for tr in range(1, N_TRIALS + 1):
    dict_ts[f"v_trial_{tr}_cmps"] = v_matrix[tr - 1]
    dict_ts[f"pos_trial_{tr}_cm"] = pos_matrix[tr - 1]

df_ts = pd.DataFrame(dict_ts)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Statistical_Summary", index=False)
    df_trials_export.to_excel(writer, sheet_name="Trial_Metrics", index=False)
    df_ts.to_excel(writer, sheet_name="Synchronized_TimeSeries", index=False)

df_ts.to_csv(csv_path, index=False)

print("=" * 85)
print(f"🎉 DATASET EXPORTED FOR MASS {MASS_HANGING_G} g!")
print(f"• Excel Workbook: {excel_path.resolve()}")
print("=" * 85)

In [ ]:
# %% [code] Cell 7: Multi-Mass Campaign Benchmark (Newton's 2nd Law - Corrected h=70cm)
data_dir = Path("data")
mass_files = list(data_dir.glob("flight_kinematics_mass_*g.xlsx"))

if len(mass_files) >= 2:
    campaign_records = []
    for f in sorted(mass_files):
        try:
            df_s = pd.read_excel(f, sheet_name="Statistical_Summary")
            campaign_records.append(df_s.iloc[0].to_dict())
        except Exception:
            pass

    df_camp = pd.DataFrame(campaign_records).sort_values("mass_hanging_g")

    h_drop_actual_m = 0.70  # ◄── Drop height: 70.0 cm
    df_camp["a_kinematic_mps2"] = (df_camp["v_max_mean_cmps"] / 100.0) ** 2 / (2.0 * h_drop_actual_m)
    df_camp["a_kinematic_sem"] = (
        2.0 * (df_camp["v_max_mean_cmps"] / 100.0) / (2.0 * h_drop_actual_m) * (df_camp["v_max_sem_cmps"] / 100.0)
    )

    m_glider_eff_kg = 0.185
    mass_ratio = (df_camp["mass_hanging_g"] / 1000.0) / (m_glider_eff_kg + (df_camp["mass_hanging_g"] / 1000.0))

    x_data = mass_ratio.to_numpy()
    y_data = df_camp["a_kinematic_mps2"].to_numpy()

    # Physically constrained zero-intercept regression: a = g * ratio
    g_constrained = float(np.sum(x_data * y_data) / np.sum(x_data ** 2))

    fig_newton = go.Figure()
    fig_newton.add_scatter(
        x=x_data, y=y_data,
        error_y=dict(type="data", array=df_camp["a_kinematic_sem"], visible=True),
        mode="markers", marker=dict(size=12, color="#00FFCC"),
        name="Measured Kinematic Accel (±1 SEM)"
    )

    x_line = np.linspace(0.0, max(x_data) * 1.15, 100)
    fig_newton.add_scatter(
        x=x_line, y=g_constrained * x_line,
        mode="lines", line=dict(color="#FFA500", width=2.8, dash="dash"),
        name=f"Physical Fit through (0,0): g_exp = {g_constrained:.2f} m/s²"
    )
    fig_newton.add_scatter(
        x=x_line, y=9.78 * x_line,
        mode="lines", line=dict(color="rgba(255, 255, 255, 0.4)", width=1.5, dash="dot"),
        name="Local Gravity Reference (g = 9.78 m/s²)"
    )

    fig_newton.update_layout(
        template="plotly_dark", height=480,
        title=f"<b>Newton's Second Law Verification: a vs. m/(M+m) (h = {h_drop_actual_m*100:.0f} cm)</b>",
        xaxis=dict(title="Theoretical Mass Ratio: m / (M + m)", range=[0.0, max(x_data) * 1.18]),
        yaxis=dict(title="Kinematic Acceleration a [m/s²]", range=[0.0, max(y_data) * 1.20]),
        margin=dict(l=60, r=30, t=55, b=35)
    )
    fig_newton.show()

    g_local = 9.78
    err_pct = abs(g_constrained - g_local) / g_local * 100.0

    print("=" * 85)
    print("🏆 FINAL MULTI-MASS EXPERIMENTAL CAMPAIGN RESULTS:")
    print("=" * 85)
    for _, row in df_camp.iterrows():
        print(f"• Mass {row['mass_hanging_g']:3.0f} g: v_max = {row['v_max_mean_cmps']:5.1f} ± {row['v_max_sem_cmps']:3.1f} cm/s | a_kin = {row['a_kinematic_mps2']:.3f} m/s² | k_bump = {row['k_bumper_mean_npm']:4.1f} N/m")
    print("-" * 85)
    print(f"• Extracted Physical Gravity (Zero-Intercept) : g_exp = {g_constrained:.2f} m/s²")
    print(f"• True Local Gravity (Colombia ~4°N)          : g_ref = {g_local:.2f} m/s²")
    print(f"• Experimental Relative Error                 : {err_pct:.1f}%")
    print("=" * 85)

## 6. Clean Hardware Teardown
Release FPGA DMA channels and memory buffers.

In [ ]:
# %% [code] Cell 8: Clean Hardware Teardown
ol.close()
print("🔒 FPGA hardware resources cleanly released.")